In [1]:
from qick import *
# %matplotlib widget
%matplotlib notebook
# %matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mpl

plt.rcParams['axes.prop_cycle'] = plt.cycler(
    color=mpl.colormaps['Set1'].colors
)

import numpy as np
from numpy.polynomial import Polynomial
import matplotlib.ticker as mtick
from matplotlib.ticker import MultipleLocator
# from tqdm import tqdm
from tqdm.notebook import tqdm
import xarray as xr

In [2]:
import os
import sys
sys.path.insert(0, '../../pattern/')
from helper_sweep import do_sweep

sys.path.insert(0, '../../instrument/')
from xilinx_qick.class_drx import drx
from xilinx_qick.class_rox import rox
from xilinx_qick.class_sweep import sweep
from xilinx_qick.instr_xilinx_v1 import XilinxProg

In [3]:
from pathlib import Path

folder_name = Path.cwd().name
data_dir = Path("Z:/labdata/qcdlabs") / folder_name
data_dir.mkdir(parents=True, exist_ok=True)

In [4]:
xilinx_1 = XilinxProg(ip_address="10.0.100.21", mode='AveragerProgram')
xilinx_1.reps = int(1)
xilinx_1.ddr4 = False
xilinx_1.mr = True
dr_ch0 = 0
ro_ch0 = 0

ro_ch1 = 1

Pyro.NameServer PYRO:Pyro.NameServer@0.0.0.0:8888
rfsoc4x2_1 PYRO:obj_3f3d0f5d2e934a07a2444f124252d9c7@10.0.100.21:33243
QICK running on RFSoC4x2, software version 0.2.381

Firmware configuration (built Wed Sep  6 18:49:29 2023):

	Global clocks (MHz): tProc dispatcher timing 409.600, RF reference 491.520
	Groups of related clocks: [tProc clock, DAC tile 0], [DAC tile 2], [ADC tile 0]

	2 signal generator channels:
	0:	axis_signal_gen_v6 - fs=9830.400 Msps, fabric=614.400 MHz
		envelope memory: 65536 complex samples (6.667 us)
		32-bit DDS, range=9830.400 MHz
		DAC tile 0, blk 0 is DAC_B
	1:	axis_signal_gen_v6 - fs=9830.400 Msps, fabric=614.400 MHz
		envelope memory: 65536 complex samples (6.667 us)
		32-bit DDS, range=9830.400 MHz
		DAC tile 2, blk 0 is DAC_A

	2 readout channels:
	0:	axis_readout_v2 - configured by PYNQ
		fs=4423.680 Msps, decimated=552.960 MHz, 32-bit DDS, range=4423.680 MHz
		axis_avg_buffer v1.0 (no edge counter, no weights)
		memory 16384 accumulated, 1024 decima

In [36]:
def set_modulation(xilinx_1,
                          carrier_frequency_hz=None,
                          modulation_frequency_hz = None,
                          modulation_depth=None):

    if carrier_frequency_hz != None:
        xilinx_1.carrier_frequency_hz = carrier_frequency_hz
    else:
        carrier_frequency_hz = xilinx_1.carrier_frequency_hz

    if modulation_frequency_hz != None:
        xilinx_1.modulation_frequency_hz = modulation_frequency_hz
    else:
        modulation_frequency_hz = xilinx_1.modulation_frequency_hz

    if modulation_depth != None:
        xilinx_1.modulation_depth = modulation_depth
    else:
        modulation_depth = xilinx_1.modulation_depth

    # print(carrier_frequency_hz, modulation_frequency_hz, modulation_depth)

    dt = 1/9830.4
    t_gen = np.arange(0, 1.6*2, dt)

    # dt = 1e-3
    # t_gen = np.arange(0, 3, dt)

    s_gen = 1 * np.exp(1j*modulation_depth*np.sin(2*np.pi*modulation_frequency_hz/1e6*t_gen))
    # s_gen = np.exp(1j*2*np.pi*modulation_frequency_hz/1e6*t_gen)

    # from scipy.special import jv
    # s_gen = jv(0, modulation_depth)
    # s_gen += jv(1, modulation_depth) * np.exp(1j*2*np.pi*modulation_frequency_hz/1e6*t_gen)
    # s_gen += -jv(1, modulation_depth) * np.exp(-1j*2*np.pi*modulation_frequency_hz/1e6*t_gen)

    dr_readout1 = drx(soc=xilinx_1.soccfg,
                    dr_ch=dr_ch0, ro_ch=ro_ch0, frequency= carrier_frequency_hz / 1e6, gain=0.9, phase=0)

    dr_readout1.wave.add(name='x2', t_data=t_gen, s_data=s_gen, idx=-1, interp_order=1)
    dr_readout1.rox.set(length=1.6, delay=1, sleep=1)
    xilinx_1.add(dr_readout1=dr_readout1)

    ro_readout2 = rox(soc=xilinx_1.soccfg, ro_ch=ro_ch1, dr_ch=dr_ch0, frequency= carrier_frequency_hz / 1e6)
    ro_readout2.set(length=1.6, delay=1, sleep=1)
    xilinx_1.add(ro_readout2=ro_readout2)

xilinx_1.register_sweep(set_modulation,
                        carrier_frequency_hz = 8.1015e9,
                        modulation_frequency_hz = 1e6,
                        modulation_depth=0.3)

In [37]:
def set_reps(xilinx_1, reps=0):
    pass

xilinx_1.register_sweep(set_reps,
                        reps=1)

In [38]:
carrier_frequency_center = 8.1056e9
carrier_frequency_span = 40e6
carrier_frequency_step = 0.25e6

carrier_frequency_list = np.arange(carrier_frequency_center-carrier_frequency_span/2, carrier_frequency_center+carrier_frequency_span/2+1, carrier_frequency_step)

# modulation_frequency_list = np.arange(0, 5e6+1, 1e6)
modulation_frequency_list = np.array([1e6, 15e6])

reps_list = np.arange(0, 2, 1)

print(len(carrier_frequency_list), len(modulation_frequency_list), len(reps_list))

161 2 2


In [39]:
config_device = [xilinx_1]
config_sweep = [
    [
        [xilinx_1.sweep, 'carrier_frequency_hz', carrier_frequency_list],
    ],
    [
        [xilinx_1.sweep, 'modulation_frequency_hz', modulation_frequency_list],
    ],
    [
        [xilinx_1.sweep, 'reps', reps_list],
    ],
]

In [40]:
_file_name = 'test_dds_modulation.zarr'
file_name= data_dir / _file_name

do_sweep(config_device, config_sweep, xilinx_1.acquire_decimated, file_name)

# xilinx_1.soc.reset_gens()

100%|██████████| 644/644 [08:25<00:00,  1.27it/s]


In [33]:
iq_mat_mr = xilinx_1.soc.get_mr()
iq_mat_mr = np.transpose(iq_mat_mr, (1, 0))
print(np.shape(iq_mat_mr))

_ticks = np.arange(0, len(iq_mat_mr[0]), 1)
_tx = xilinx_1.soc.cycles2us(_ticks, ro_ch=0)/8

iq_mat_mr = iq_mat_mr[0] + 1j* iq_mat_mr[1]

# plt.figure(figsize=(30,5))
# plt.plot(_tx, iq_mat_mr[0])
# plt.plot(_tx, iq_mat_mr[1])

# s_0 = np.exp(-1j*2*np.pi*carrier_frequency_list[-1]/1e6*_tx)
# iq_0 = iq_mat_mr * s_0
#
# s_1 = np.exp(-1j*2*np.pi*(carrier_frequency_list[-1]+modulation_frequency_list[-1])/1e6*_tx)
# iq_1 = iq_mat_mr * s_1
#
# s_2 = np.exp(-1j*2*np.pi*(carrier_frequency_list[-1]-modulation_frequency_list[-1])/1e6*_tx)
# iq_2 = iq_mat_mr * s_2
#
# M = len(iq_mat_mr) // 8
#
# _tx = _tx[:M*8].reshape(M, 8).mean(axis=1)
# iq_0 = iq_0[:M*8].reshape(M, 8).mean(axis=1)
# iq_1 = iq_1[:M*8].reshape(M, 8).mean(axis=1)
# iq_2 = iq_2[:M*8].reshape(M, 8).mean(axis=1)
#
fig = plt.figure(figsize=(12, 6))
gs = fig.add_gridspec(2, 1)

ax0 = fig.add_subplot(gs[0])
ax1 = fig.add_subplot(gs[1])

ax0.plot(_tx, np.real(iq_mat_mr))
ax1.plot(_tx, np.abs(iq_mat_mr)**2)

ax1.plot(_tx, -1e7*np.cos(2*np.pi*modulation_frequency_list[-1]/1e6*_tx))

# ax0.plot(_tx, np.real(iq_0))
# ax1.plot(_tx, np.imag(iq_0))
#
# ax0.plot(_tx, np.real(iq_1))
# ax1.plot(_tx, np.imag(iq_1))
#
# ax0.plot(_tx, np.real(iq_2))
# ax1.plot(_tx, np.imag(iq_2))

plt.show()

(2, 8184)


<IPython.core.display.Javascript object>

In [43]:
_file_name = 'test_dds_modulation.zarr'

with xr.open_zarr(data_dir / _file_name) as f:
    iq_mat = f['IQ mr']

fig = plt.figure(figsize=(12, 6))
gs = fig.add_gridspec(2, 2)

modulation_frequency_hz = 15e6

for _rox in iq_mat.rfsoc4x2_1_rox:
    ax0 = fig.add_subplot(gs[0,_rox])
    ax1 = fig.add_subplot(gs[1,_rox])

    s_data = iq_mat.sel(rfsoc4x2_1_rox=_rox, rfsoc4x2_1_modulation_frequency_hz=modulation_frequency_hz).squeeze()
    t_exp = s_data.tx

    p_mean = (s_data * np.conj(s_data) ).mean(dim='rfsoc4x2_1_reps')

    # s_mean = s_data.mean(dim='rfsoc4x2_1_reps')
    # p_mean = s_mean * np.conj(s_mean)

    plot_carrier_frequency = p_mean.rfsoc4x2_1_carrier_frequency_hz.data/1e9

    # s_demod = np.exp(1j * 2*np.pi * modulation_frequency_hz/1e6 * t_exp)
    # s_pdh = s_demod * p_mean

    fs_dec = 552.960e6
    total_N = p_mean.sizes['rfsoc4x2_1_ticks']

    if modulation_frequency_hz == 0:
        idx = total_N
    else:
        n_cycles = int(total_N * modulation_frequency_hz / fs_dec)
        idx = int(round(n_cycles * fs_dec / modulation_frequency_hz))
        idx = min(idx, total_N)

    # idx = np.searchsorted(p_mean.tx.values, t_int, side='right')

    s_demod = np.exp(1j * 2*np.pi * modulation_frequency_hz/1e6 * t_exp)
    s_demod[idx:] = 0
    s_int = (s_demod * p_mean).mean(dim='rfsoc4x2_1_ticks')

    # s_int = s_pdh.isel(rfsoc4x2_1_ticks=slice(0, idx)).mean(dim='rfsoc4x2_1_ticks')
    # s_int = s_pdh.isel(rfsoc4x2_1_ticks=slice(0, idx)).integrate('tx')

    # s_int = s_pdh.where(s_pdh.tx <= t_int, drop=True).integrate('tx')

    re_pdh = np.real(s_int)
    im_pdh = np.imag(s_int)

    ax0.plot(plot_carrier_frequency, re_pdh, '.-', color='C0')
    ax0.plot(plot_carrier_frequency, im_pdh, '.-', color='C1')

    am_pdh = np.abs(s_int)
    ph_pdh = np.angle(s_int)

    ax1.plot(plot_carrier_frequency, ph_pdh)

    # ax0.vlines((8.1056e9-modulation_frequency_hz)/1e9, np.min(im_pdh), np.max(im_pdh), color='k')
    # ax0.vlines((8.1056e9+modulation_frequency_hz)/1e9, np.min(im_pdh), np.max(im_pdh), color='k')

plt.show()

Traceback (most recent call last):
  File "C:\Users\qcduser\anaconda3\envs\QCDLabs\Lib\site-packages\xarray\core\dataset.py", line 1231, in _construct_dataarray
    variable = self._variables[name]
               ~~~~~~~~~~~~~~~^^^^^^
KeyError: 'IQ mr'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\qcduser\anaconda3\envs\QCDLabs\Lib\site-packages\xarray\core\dataset.py", line 1338, in __getitem__
    return self._construct_dataarray(key)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\qcduser\anaconda3\envs\QCDLabs\Lib\site-packages\xarray\core\dataset.py", line 1233, in _construct_dataarray
    _, name, variable = _get_virtual_variable(self._variables, name, self.sizes)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\qcduser\anaconda3\envs\QCDLabs\Lib\site-packages\xarray\core\dataset_utils.py", line 79, in _get_virtual_variable
    raise KeyError(key

In [11]:
iq_mat_ddr4 = xilinx_1.soc.get_ddr4(10)
iq_mat_ddr4 = np.transpose(iq_mat_ddr4, (1, 0))
print(np.shape(iq_mat_ddr4))

_ticks = np.arange(0, len(iq_mat_ddr4[0]), 1)
_tx = xilinx_1.soc.cycles2us(_ticks, ro_ch=0)

plt.figure(figsize=(30,5))
plt.plot(_tx, iq_mat_ddr4[0])
plt.plot(_tx, iq_mat_ddr4[1])


_file_name = 'test_dds_modulation.zarr'

with xr.open_zarr(data_dir / _file_name) as f:
    iq_mat = f['IQ ddr4']

# fig = plt.figure(figsize=(12, 6))
# gs = fig.add_gridspec(2, 2)

_rox = 0
modulation_frequency_hz = 15e6

s_data = iq_mat.sel(rfsoc4x2_1_rox=_rox, rfsoc4x2_1_carrier_frequency_hz=carrier_frequency_list[-1], rfsoc4x2_1_modulation_frequency_hz=modulation_frequency_hz, rfsoc4x2_1_reps=1).squeeze()
t_exp = s_data.tx

plt.scatter(t_exp, np.real(s_data))
plt.scatter(t_exp, np.imag(s_data))


plt.show()

(2, 1759)


<IPython.core.display.Javascript object>

In [ ]:
_file_name = 'test_dds_modulation.zarr'

with xr.open_zarr(data_dir / _file_name) as f:
    iq_mat = f['IQ decimated']

fig = plt.figure(figsize=(12, 6))
gs = fig.add_gridspec(2, 2)

_rox = 0
modulation_frequency_hz = 15e6

s_data = iq_mat.sel(rfsoc4x2_1_rox=_rox, rfsoc4x2_1_carrier_frequency_hz=carrier_frequency_list[-1], rfsoc4x2_1_modulation_frequency_hz=modulation_frequency_hz, rfsoc4x2_1_reps=1).squeeze()
t_exp = s_data.tx

plt.figure(figsize=(30,5))
plt.plot(t_exp, np.real(s_data))
plt.plot(t_exp, np.imag(s_data))
plt.show()

# for _rox in iq_mat.rfsoc4x2_1_rox:
#     ax0 = fig.add_subplot(gs[0,_rox])
#     ax1 = fig.add_subplot(gs[1,_rox])
#
#     s_data = iq_mat.sel(rfsoc4x2_1_rox=_rox, rfsoc4x2_1_modulation_frequency_hz=modulation_frequency_hz).squeeze()
#     t_exp = s_data.tx
#
#     p_mean = (s_data * np.conj(s_data) ).mean(dim='rfsoc4x2_1_reps')
#
#     # s_mean = s_data.mean(dim='rfsoc4x2_1_reps')
#     # p_mean = s_mean * np.conj(s_mean)
#
#     plot_carrier_frequency = p_mean.rfsoc4x2_1_carrier_frequency_hz.data/1e9
#
#     # s_demod = np.exp(1j * 2*np.pi * modulation_frequency_hz/1e6 * t_exp)
#     # s_pdh = s_demod * p_mean
#
#     if modulation_frequency_hz == 0:
#         t_int = p_mean.tx[-1]
#     else:
#         T_mod = 1e6/modulation_frequency_hz
#         # N_period = 1
#         N_period = int(p_mean.tx[-1] // T_mod)
#         t_int = N_period * T_mod
#
#     idx = np.searchsorted(p_mean.tx.values, t_int, side='right')
#
#     s_demod = np.exp(1j * 2*np.pi * modulation_frequency_hz/1e6 * t_exp)
#     s_demod[idx:] = 0
#     s_int = (s_demod * p_mean).mean(dim='rfsoc4x2_1_ticks')
#
#     # s_int = s_pdh.isel(rfsoc4x2_1_ticks=slice(0, idx)).mean(dim='rfsoc4x2_1_ticks')
#     # s_int = s_pdh.isel(rfsoc4x2_1_ticks=slice(0, idx)).integrate('tx')
#
#     # s_int = s_pdh.where(s_pdh.tx <= t_int, drop=True).integrate('tx')
#
#     re_pdh = np.real(s_int)
#     im_pdh = np.imag(s_int)
#
#     ax0.plot(plot_carrier_frequency, re_pdh, '.-', color='C0')
#     ax0.plot(plot_carrier_frequency, im_pdh, '.-', color='C1')
#
#     am_pdh = np.abs(s_int)
#     ph_pdh = np.angle(s_int)
#
#     ax1.plot(plot_carrier_frequency, ph_pdh)
#
#     # ax0.vlines((8.1056e9-modulation_frequency_hz)/1e9, np.min(im_pdh), np.max(im_pdh), color='k')
#     # ax0.vlines((8.1056e9+modulation_frequency_hz)/1e9, np.min(im_pdh), np.max(im_pdh), color='k')
#
# plt.show()

In [ ]:
s_data.coords

In [ ]:
_file_name = 'test_dds_modulation.zarr'

with xr.open_zarr(data_dir / _file_name) as f:
    iq_mat = f['IQ decimated']

fig = plt.figure(figsize=(12, 6))
gs = fig.add_gridspec(2, 2)

modulation_frequency_hz = 15e6

for _rox in iq_mat.rfsoc4x2_1_rox:
    ax0 = fig.add_subplot(gs[0,_rox])
    ax1 = fig.add_subplot(gs[1,_rox])

    s_data = iq_mat.sel(rfsoc4x2_1_rox=_rox, rfsoc4x2_1_modulation_frequency_hz=modulation_frequency_hz).squeeze()
    t_exp = s_data.tx

    p_mean = (s_data * np.conj(s_data) ).mean(dim='rfsoc4x2_1_reps')

    # s_mean = s_data.mean(dim='rfsoc4x2_1_reps')
    # p_mean = s_mean * np.conj(s_mean)

    plot_carrier_frequency = p_mean.rfsoc4x2_1_carrier_frequency_hz.data/1e9

    # s_demod = np.exp(1j * 2*np.pi * modulation_frequency_hz/1e6 * t_exp)
    # s_pdh = s_demod * p_mean

    if modulation_frequency_hz == 0:
        t_int = p_mean.tx[-1]
    else:
        T_mod = 1e6/modulation_frequency_hz
        # N_period = 1
        N_period = int(p_mean.tx[-1] // T_mod)
        t_int = N_period * T_mod

    idx = np.searchsorted(p_mean.tx.values, t_int, side='right')

    s_demod = np.exp(1j * 2*np.pi * modulation_frequency_hz/1e6 * t_exp)
    s_demod[idx:] = 0
    s_int = (s_demod * p_mean).mean(dim='rfsoc4x2_1_ticks')

    # s_int = s_pdh.isel(rfsoc4x2_1_ticks=slice(0, idx)).mean(dim='rfsoc4x2_1_ticks')
    # s_int = s_pdh.isel(rfsoc4x2_1_ticks=slice(0, idx)).integrate('tx')

    # s_int = s_pdh.where(s_pdh.tx <= t_int, drop=True).integrate('tx')

    re_pdh = np.real(s_int)
    im_pdh = np.imag(s_int)

    ax0.plot(plot_carrier_frequency, re_pdh, '.-', color='C0')
    ax0.plot(plot_carrier_frequency, im_pdh, '.-', color='C1')

    am_pdh = np.abs(s_int)
    ph_pdh = np.angle(s_int)

    ax1.plot(plot_carrier_frequency, ph_pdh)

    # ax0.vlines((8.1056e9-modulation_frequency_hz)/1e9, np.min(im_pdh), np.max(im_pdh), color='k')
    # ax0.vlines((8.1056e9+modulation_frequency_hz)/1e9, np.min(im_pdh), np.max(im_pdh), color='k')

plt.show()

In [ ]:
_file_name = 'test_dds_modulation.zarr'

with xr.open_zarr(data_dir / _file_name) as f:
    iq_mat = f['IQ decimated']

fig = plt.figure(figsize=(12, 6))
gs = fig.add_gridspec(2, 2)

modulation_frequency_hz = 5e6

for _rox in iq_mat.rfsoc4x2_1_rox[:]:
    ax0 = fig.add_subplot(gs[0,_rox])
    ax1 = fig.add_subplot(gs[1,_rox])

    s_data = iq_mat.sel(rfsoc4x2_1_rox=_rox, rfsoc4x2_1_modulation_frequency_hz=modulation_frequency_hz).squeeze()
    t_exp = s_data.tx

    p_mean = (s_data * np.conj(s_data) ).mean(dim='rfsoc4x2_1_reps')

    # s_mean = s_data.mean(dim='rfsoc4x2_1_reps')
    # p_mean = s_mean * np.conj(s_mean)

    plot_carrier_frequency = p_mean.rfsoc4x2_1_carrier_frequency_hz.data/1e9

    # s_demod = np.exp(1j * 2*np.pi * modulation_frequency_hz/1e6 * t_exp)
    # s_pdh = s_demod * p_mean

    if modulation_frequency_hz == 0:
        t_int = p_mean.tx[-1]
    else:
        T_mod = 1e6/modulation_frequency_hz
        N_period = int(p_mean.tx[-1] // T_mod)
        t_int = N_period * T_mod

    idx = np.searchsorted(p_mean.tx.values, t_int, side='right')

    s_demod = np.exp(1j * 2*np.pi * modulation_frequency_hz/1e6 * t_exp)
    s_demod[idx:] = 0
    s_int = (s_demod * p_mean).mean(dim='rfsoc4x2_1_ticks')

    re_pdh = np.real(s_int)
    im_pdh = np.imag(s_int)

    # ax0.plot(plot_carrier_frequency, re_pdh, '-', color='C0')
    # ax0.plot(plot_carrier_frequency, im_pdh, '-', color='C1')

    from scipy.signal import savgol_filter, find_peaks
    dphase = (re_pdh/im_pdh).data.compute()

    dphase2 = np.gradient(dphase, plot_carrier_frequency)
    peaks_phase, props = find_peaks(dphase2,
                                    distance=3)

    # peaks_phase = np.where(np.abs(dphase2) > 2000)[0]

    all_idx = np.arange(len(plot_carrier_frequency))
    rest_idx = np.setdiff1d(all_idx, peaks_phase)

    ax0.plot(plot_carrier_frequency[peaks_phase], re_pdh[peaks_phase], '.-', color='k', alpha=0.3)
    # ax0.plot(plot_carrier_frequency[peaks_phase], im_pdh[peaks_phase], '.-', color='k', alpha=0.3)

    ax0.plot(plot_carrier_frequency[rest_idx], re_pdh[rest_idx], '.-', color='C0', alpha=0.3)
    # ax0.plot(plot_carrier_frequency[rest_idx], im_pdh[rest_idx], '.-', color='C1', alpha=0.3)

    ax1.plot(plot_carrier_frequency, dphase2, '.-', color='C0', alpha=0.3)
    ax1.set_ylim(-1000,1000)

    # ax1.scatter(re_pdh[peaks_phase], im_pdh[peaks_phase], color='k', alpha=0.3)
    # ax1.scatter(re_pdh[rest_idx], im_pdh[rest_idx], color='C0', alpha=0.3)

plt.show()

In [ ]:
_file_name = 'test_dds_modulation.zarr'

with xr.open_zarr(data_dir / _file_name) as f:
    iq_mat = f['IQ decimated'].compute()

fig = plt.figure(figsize=(12, 6))
gs = fig.add_gridspec(2, 2)

modulation_frequency_hz = 0e6

for _rox in iq_mat.rfsoc4x2_1_rox:
    ax0 = fig.add_subplot(gs[0,_rox])
    ax1 = fig.add_subplot(gs[1,_rox])

    s_data = iq_mat.sel(rfsoc4x2_1_rox=_rox, rfsoc4x2_1_modulation_depth=0.3, rfsoc4x2_1_modulation_frequency_hz=modulation_frequency_hz).squeeze()
    t_exp = s_data.tx

    # s_int = s_data.mean(dim='rfsoc4x2_1_ticks')
    plot_carrier_frequency = s_data.rfsoc4x2_1_carrier_frequency_hz.data/1e9

    abs_data = xr.apply_ufunc(np.abs, s_data)
    # abs_mean = xr.apply_ufunc(np.abs, s_data.mean(dim=['rfsoc4x2_1_ticks', 'rfsoc4x2_1_reps']))
    abs_mean = abs_data.mean(dim=['rfsoc4x2_1_ticks', 'rfsoc4x2_1_reps'])
    abs_std = abs_data.std(dim=['rfsoc4x2_1_ticks', 'rfsoc4x2_1_reps'])

    ph_data = xr.apply_ufunc(np.angle,s_data)
    # ph_data = xr.apply_ufunc(np.unwrap,ph_data)

    ph_mean = xr.apply_ufunc(np.angle, s_data.mean(dim=['rfsoc4x2_1_ticks', 'rfsoc4x2_1_reps']))
    # ph_mean = s_int.mean(dim=['rfsoc4x2_1_ticks', 'rfsoc4x2_1_reps'])
    ph_std = ph_data.std(dim=['rfsoc4x2_1_ticks', 'rfsoc4x2_1_reps'])

    coef = np.polyfit(plot_carrier_frequency, np.unwrap(ph_mean), 1)
    ph_fit = np.polyval(coef, plot_carrier_frequency)

    ax0.plot(plot_carrier_frequency, abs_mean, '.-', color='C2')
    ax0.fill_between(plot_carrier_frequency,
                     abs_mean-abs_std,
                     abs_mean+abs_std, alpha=0.3, color='C0')

    ax1.plot(plot_carrier_frequency, np.unwrap(ph_mean)-ph_fit, '.-', color='C2')
    ax1.fill_between(plot_carrier_frequency, np.unwrap(ph_mean)-ph_fit-ph_std, np.unwrap(ph_mean)-ph_fit+ph_std, alpha=0.3, color='C0')

plt.show()

In [ ]:
_file_name = 'test_dds_modulation.zarr'

with xr.open_zarr(data_dir / _file_name) as f:
    iq_mat = f['IQ decimated'].load()

fig = plt.figure(figsize=(12, 6))
gs = fig.add_gridspec(2, 2)

for _rox in iq_mat.rfsoc4x2_1_rox:
    ax0 = fig.add_subplot(gs[0,_rox])
    ax1 = fig.add_subplot(gs[1,_rox])

    s_data = iq_mat.sel(rfsoc4x2_1_rox=_rox, rfsoc4x2_1_modulation_depth=0.3).squeeze()
    ticks = s_data.tx

    carrier_frequency_hz = s_data.rfsoc4x2_1_carrier_frequency_hz.data[1]
    s_trace = s_data.sel(rfsoc4x2_1_carrier_frequency_hz=carrier_frequency_hz, rfsoc4x2_1_modulation_frequency_hz=5e6)

    abs_data = xr.apply_ufunc(np.abs, s_trace)
    # abs_mean = xr.apply_ufunc(np.abs, s_trace.mean(dim='rfsoc4x2_1_reps'))
    abs_mean = abs_data.mean(dim='rfsoc4x2_1_reps')
    abs_std = abs_data.std(dim='rfsoc4x2_1_reps')

    ph_data = xr.apply_ufunc(np.angle,s_trace)
    ph_mean = xr.apply_ufunc(np.angle, s_trace.mean(dim='rfsoc4x2_1_reps'))
    # ph_mean = ph_data.mean(dim='rfsoc4x2_1_reps')
    ph_std = ph_data.std(dim='rfsoc4x2_1_reps')

    ax0.plot(ticks, abs_mean, '.-', color='C2')
    ax0.fill_between(ticks,
                     abs_mean-abs_std,
                     abs_mean+abs_std, alpha=0.3, color='C0')

    ax1.plot(ticks, ph_mean, '.-', color='C2')
    ax1.fill_between(ticks, ph_mean-ph_std, ph_mean+ph_std, alpha=0.3, color='C0')

# ax0.set_xlim(8.100, 8.1043)
# ax1.set_xlim(8.100, 8.1043)

plt.show()


    # plt.figure()
# plt.plot(xilinx_1.config['dr_readout1'].wave.items[0].i_data, '.-')
# plt.plot(xilinx_1.config['dr_readout1'].wave.items[0].q_data, '.-')
# plt.show()

In [ ]:
_file_name = 'test_dds_modulation.zarr'

with xr.open_zarr(data_dir / _file_name) as f:
    iq_mat = f['IQ decimated'].load()

fig = plt.figure(figsize=(12, 6))
gs = fig.add_gridspec(2, 2)

for _rox in iq_mat.rfsoc4x2_1_rox:
    ax0 = fig.add_subplot(gs[0,_rox])
    ax1 = fig.add_subplot(gs[1,_rox])

    s_data = iq_mat.sel(rfsoc4x2_1_rox=_rox, rfsoc4x2_1_modulation_depth=1, rfsoc4x2_1_modulation_frequency_hz=1e6).squeeze()
    # t_exp = s_data.tx

    s_int = s_data.mean(dim='rfsoc4x2_1_ticks')
    plot_carrier_frequency = s_int.rfsoc4x2_1_carrier_frequency_hz.data/1e9

    abs_data = xr.apply_ufunc(np.abs, s_int)
    abs_mean = xr.apply_ufunc(np.abs, s_int.mean(dim='rfsoc4x2_1_reps'))
    # abs_mean = abs_data.mean(dim='rfsoc4x2_1_reps')
    abs_std = abs_data.std(dim='rfsoc4x2_1_reps')

    ph_data = xr.apply_ufunc(np.angle,s_int)
    # ph_data = xr.apply_ufunc(np.unwrap,ph_data)

    ph_mean = xr.apply_ufunc(np.angle, s_int.mean(dim='rfsoc4x2_1_reps'))
    # ph_mean = s_int.mean(dim='rfsoc4x2_1_reps')
    ph_std = ph_data.std(dim='rfsoc4x2_1_reps')

    coef = np.polyfit(plot_carrier_frequency, np.unwrap(ph_mean), 1)
    ph_fit = np.polyval(coef, plot_carrier_frequency)
    # ph_fit = np.angle(np.exp(1j * ph_fit))

    ax0.plot(plot_carrier_frequency, abs_mean, '.-', color='C2')
    # ax0.fill_between(plot_carrier_frequency,
    #                  abs_mean-abs_std,
    #                  abs_mean+abs_std, alpha=0.3, color='C0')

    ax1.plot(plot_carrier_frequency, np.unwrap(ph_mean)-ph_fit, '.-', color='C2')
    ax1.fill_between(plot_carrier_frequency, np.unwrap(ph_mean)-ph_fit-ph_std, np.unwrap(ph_mean)-ph_fit+ph_std, alpha=0.3, color='C0')

# ax0.set_xlim(8.100, 8.1043)
# ax1.set_xlim(8.100, 8.1043)

plt.show()

In [ ]:
_file_name = 'test_dds_modulation.zarr'

with xr.open_zarr(data_dir / _file_name) as f:
    iq_mat = f['IQ decimated'].load()

fig = plt.figure(figsize=(12, 6))
gs = fig.add_gridspec(2, 2)

modulation_frequency_hz = 1e6

for _rox in iq_mat.rfsoc4x2_1_rox:
    ax0 = fig.add_subplot(gs[0,_rox])
    ax1 = fig.add_subplot(gs[1,_rox])

    s_data = iq_mat.sel(rfsoc4x2_1_rox=_rox, rfsoc4x2_1_modulation_depth=1, rfsoc4x2_1_modulation_frequency_hz=modulation_frequency_hz).squeeze()
    t_exp = s_data.tx

    p_mean = (s_data * np.conj(s_data) ).mean(dim='rfsoc4x2_1_reps')
    plot_carrier_frequency = p_mean.rfsoc4x2_1_carrier_frequency_hz.data/1e9

    s_demod = np.exp(-1j * 2*np.pi * modulation_frequency_hz/1e6 * t_exp)
    s_pdh = s_demod * p_mean

    t_int = 1 * (1e6/s_data.rfsoc4x2_1_modulation_frequency_hz)
    s_int = s_pdh.where(s_pdh.tx <= t_int, drop=True).integrate('tx')

    re_pdh = np.real(s_int)
    im_pdh = np.imag(s_int)

    ax0.plot(plot_carrier_frequency, re_pdh, '.-', color='C2')
    ax1.plot(plot_carrier_frequency, im_pdh, '.-', color='C2')

plt.show()